# MOD-02 · NB-07 — Geospatial Health Disparities Map
### Health Analytics with Python · Module 02: EDA in Healthcare
---
**Learning objectives**
- Join patient ZIP codes to census tracts using synthetic spatial data
- Compute age-adjusted hospitalisation rates and readmission rates by area
- Create choropleth maps with `geopandas` and `matplotlib`
- Overlay a social vulnerability index (SVI) and compute spatial correlations
- Export interactive maps with `folium`

**Estimated time:** 3 hours | **Level:** Advanced | **Libraries:** `pandas`, `numpy`, `geopandas`, `matplotlib`, `folium`, `shapely`


## 1. Setup & Dependency Check

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

# Optional geospatial libraries
geo_available = False
try:
    import geopandas as gpd
    from shapely.geometry import Point, Polygon
    geo_available = True
    print(f"geopandas {gpd.__version__} available.")
except ImportError:
    print("geopandas not found — install with: pip install geopandas shapely")
    print("Continuing with matplotlib-only spatial visualisation.")

folium_available = False
try:
    import folium
    from folium.plugins import HeatMap
    folium_available = True
    print(f"folium {folium.__version__} available.")
except ImportError:
    print("folium not found — install with: pip install folium")

plt.rcParams.update({'figure.dpi':120,'figure.facecolor':'white'})


## 2. Synthetic Geospatial Dataset — 50 ZIP Code Areas

In [ ]:
np.random.seed(42)

# ── Simulate 50 ZIP code areas (modelled on a mid-size US metropolitan area) ──
N_ZIPS = 50

# Geographic centroids (roughly spanning a metro area, lat/lon)
lat_center, lon_center = 43.0, -76.1   # ~ Upstate NY
zip_df = pd.DataFrame({
    'zip'           : [f'{10000+i:05d}' for i in range(N_ZIPS)],
    'tract_name'    : [f'Tract {100+i}' for i in range(N_ZIPS)],
    'lat'           : np.random.uniform(lat_center-0.5, lat_center+0.5, N_ZIPS),
    'lon'           : np.random.uniform(lon_center-0.5, lon_center+0.5, N_ZIPS),
    'population'    : np.random.randint(3000, 25000, N_ZIPS),
    'pct_poverty'   : np.random.beta(2,8,N_ZIPS)*50,     # 0-50%
    'pct_uninsured' : np.random.beta(1.5,8,N_ZIPS)*30,
    'median_income' : np.random.normal(52000,18000,N_ZIPS).clip(15000,120000),
    'svi_score'     : np.random.beta(2,3,N_ZIPS),        # CDC SVI 0–1, higher=worse
    'pct_elderly'   : np.random.beta(2,6,N_ZIPS)*40,     # % aged 65+
    'urban_rural'   : np.random.choice(['Urban','Suburban','Rural'],N_ZIPS,p=[0.50,0.35,0.15])
})

# Hospitalisations inversely correlated with income and SVI
zip_df['hosp_rate'] = (
    200
    + zip_df['svi_score'] * 180
    - zip_df['median_income']/1000 * 1.5
    + zip_df['pct_elderly'] * 4.5
    + np.random.normal(0,25,N_ZIPS)
).clip(50,600)

# Readmission rate
zip_df['readmit_rate'] = (
    12
    + zip_df['svi_score'] * 10
    + zip_df['pct_poverty'] * 0.18
    + np.random.normal(0,1.5,N_ZIPS)
).clip(5,35)

# Age-adjusted rate (add noise to crude to simulate adjustment)
zip_df['hosp_rate_adj'] = (zip_df['hosp_rate']
                           - zip_df['pct_elderly']*3.2
                           + np.random.normal(0,15,N_ZIPS)).clip(50,600)

print(f"ZIP code dataset: {zip_df.shape}")
display(zip_df.head())


## 3. Patient-to-ZIP Assignment & Aggregation

In [ ]:
# ── Generate patient discharge records with ZIP codes ─────────────────────────
np.random.seed(99); N = 800
# Assign patients to ZIPs weighted by population
zip_weights = zip_df['population'] / zip_df['population'].sum()
patient_zips = np.random.choice(zip_df['zip'], size=N, p=zip_weights)

patients = pd.DataFrame({
    'patient_id'   : [f'PT-{str(i).zfill(5)}' for i in range(1,N+1)],
    'zip'          : patient_zips,
    'age'          : np.random.normal(62,16,N).clip(18,95).astype(int),
    'sex'          : np.random.choice(['M','F'],N,p=[0.48,0.52]),
    'los_days'     : np.random.gamma(2.5,1.8,N).clip(1,30).astype(int),
    'readmitted_30': np.random.binomial(1,0.14,N),
    'total_charge' : (np.random.gamma(2.5,1.8,N)*2500).round(2),
})

# Aggregate to ZIP level
zip_agg = (
    patients.groupby('zip')
            .agg(n_discharges    = ('patient_id','count'),
                 n_readmitted    = ('readmitted_30','sum'),
                 mean_los        = ('los_days','mean'),
                 mean_charge     = ('total_charge','mean'))
            .reset_index()
)
zip_agg['observed_readmit_rate'] = zip_agg['n_readmitted']/zip_agg['n_discharges']*100

# Merge with ZIP reference table
zip_merged = zip_df.merge(zip_agg, on='zip', how='left')
zip_merged[['n_discharges','n_readmitted']] = zip_merged[['n_discharges','n_readmitted']].fillna(0)
print(f"Merged ZIP dataset: {zip_merged.shape}")
print(f"ZIPs with ≥5 discharges: {(zip_merged['n_discharges']>=5).sum()}")


## 4. Scatter-Map Choropleth (matplotlib — no geopandas required)

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(16,6))
fig.suptitle('Geographic distribution of health outcomes', fontsize=13)

for ax, metric, cmap, title, fmt in [
        (axes[0],'hosp_rate_adj','YlOrRd',
         'Age-adjusted hospitalisation rate (per 100,000 population)','{:.0f}'),
        (axes[1],'readmit_rate','Blues',
         '30-day readmission rate (%)','{:.1f}%'),
]:
    sc = ax.scatter(
        zip_merged['lon'], zip_merged['lat'],
        c=zip_merged[metric], cmap=cmap,
        s=zip_merged['population']/100 + 20,
        alpha=0.8, edgecolors='white', linewidth=0.5
    )
    cbar = plt.colorbar(sc, ax=ax, shrink=0.75)
    cbar.set_label(metric.replace('_',' '))

    # Annotate urban/rural
    for urb, marker in [('Urban','o'),('Suburban','s'),('Rural','^')]:
        mask = zip_merged['urban_rural']==urb
        ax.scatter([],[], marker=marker, color='gray', s=50, label=urb, alpha=0.7)
    ax.legend(title='Area type', fontsize=8, loc='lower left')

    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_title(title, fontsize=10)
    ax.set_facecolor('#F0F4F8')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/mod02/geo_scatter_maps.png', bbox_inches='tight')
plt.show()


## 5. SVI Correlation Analysis

In [ ]:
# ── Pearson and Spearman correlations with SVI ────────────────────────────────
outcomes = ['hosp_rate_adj','readmit_rate','mean_los','mean_charge']
outcome_labels = ['Adj. hosp. rate','Readmit. rate (%)','Mean LOS','Mean charge ($)']

print(f"{'Outcome':20s}  {'Pearson r':>10s}  {'p':>7s}  {'Spearman ρ':>10s}  {'p':>7s}")
print("-"*65)
for col, label in zip(outcomes, outcome_labels):
    sub = zip_merged[['svi_score',col]].dropna()
    r_p, p_p = pearsonr(sub['svi_score'], sub[col])
    r_s, p_s = spearmanr(sub['svi_score'], sub[col])
    print(f"{label:20s}  {r_p:>+10.3f}  {p_p:>7.4f}  {r_s:>+10.3f}  {p_s:>7.4f}")


In [ ]:
# ── Scatter plots: SVI vs health outcomes ────────────────────────────────────
fig, axes = plt.subplots(1,2,figsize=(13,5))

color_urban = {'Urban':'#4878CF','Suburban':'#6ACC65','Rural':'#D65F5F'}

for ax, metric, ylabel in [
        (axes[0],'hosp_rate_adj','Age-adjusted hospitalisation rate'),
        (axes[1],'readmit_rate', '30-day readmission rate (%)')
]:
    for urb, color in color_urban.items():
        sub = zip_merged[zip_merged['urban_rural']==urb]
        ax.scatter(sub['svi_score'], sub[metric],
                   color=color, alpha=0.7, s=sub['population']/150+15,
                   label=urb, edgecolors='white', lw=0.5)
    # OLS trend line
    m, b = np.polyfit(zip_merged['svi_score'].dropna(), zip_merged[metric].dropna(), 1)
    x_range = np.linspace(0,1,50)
    ax.plot(x_range, m*x_range+b, 'k--', lw=1.3, label='OLS trend')
    r, p = pearsonr(zip_merged['svi_score'].dropna(), zip_merged[metric].dropna())
    ax.text(0.04, 0.95, f'r = {r:.2f}, p = {p:.3f}',
            transform=ax.transAxes, fontsize=10, va='top',
            bbox=dict(facecolor='white',alpha=0.8,edgecolor='none'))
    ax.set_xlabel('Social Vulnerability Index (SVI)')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} vs SVI')
    ax.legend(fontsize=9, title='Area type')

plt.tight_layout()
plt.savefig('/tmp/mod02/svi_scatter.png', bbox_inches='tight')
plt.show()


## 6. Quintile Disparity Analysis

In [ ]:
# Stratify ZIP codes into SVI quintiles and compare outcomes
zip_merged['svi_quintile'] = pd.qcut(zip_merged['svi_score'],5,
                                      labels=['Q1 (Least)','Q2','Q3','Q4','Q5 (Most)'])

disparity_table = (
    zip_merged.groupby('svi_quintile',observed=True)
              .agg(
                  n_zips             = ('zip','count'),
                  mean_population    = ('population','mean'),
                  hosp_rate          = ('hosp_rate_adj','mean'),
                  readmit_rate       = ('readmit_rate','mean'),
                  pct_uninsured      = ('pct_uninsured','mean'),
                  median_income      = ('median_income','mean'),
              )
              .round(1)
)
print("Health outcomes by SVI quintile:")
display(disparity_table)

ratio = (disparity_table['hosp_rate'].iloc[-1] /
         disparity_table['hosp_rate'].iloc[0])
print(f"\nDisparity ratio (Q5 vs Q1) for hospitalisation rate: {ratio:.2f}×")


In [ ]:
# Bar chart of outcomes by SVI quintile
fig, axes = plt.subplots(1,2,figsize=(13,5))
colors_q = ['#2166ac','#74add1','#fee090','#f46d43','#a50026']

for ax, metric, ylabel in [
        (axes[0],'hosp_rate','Adj. hosp. rate per 100k'),
        (axes[1],'readmit_rate','Readmission rate (%)')
]:
    vals = disparity_table[metric].values
    bars = ax.bar(disparity_table.index, vals, color=colors_q, edgecolor='white', lw=0.5)
    for bar,val in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2, val+1,
                f'{val:.0f}', ha='center', va='bottom', fontsize=9)
    ax.set_xlabel('SVI quintile'); ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} by SVI quintile')
    ax.tick_params(axis='x',rotation=15)

plt.suptitle('Health disparities across social vulnerability quintiles', fontsize=12)
plt.tight_layout()
plt.savefig('/tmp/mod02/svi_quintile_bar.png', bbox_inches='tight')
plt.show()


## 7. Folium Interactive Map

In [ ]:
if folium_available:
    m = folium.Map(location=[lat_center, lon_center], zoom_start=10, tiles='CartoDB positron')

    # Choropleth via CircleMarker (no shapefile needed)
    cmap_fn = plt.get_cmap('YlOrRd')
    norm    = mcolors.Normalize(vmin=zip_merged['hosp_rate_adj'].min(),
                                 vmax=zip_merged['hosp_rate_adj'].max())

    for _, row in zip_merged.dropna(subset=['hosp_rate_adj']).iterrows():
        color = mcolors.to_hex(cmap_fn(norm(row['hosp_rate_adj'])))
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=max(4, row['n_discharges']/5),
            color='white', weight=0.5,
            fill=True, fill_color=color, fill_opacity=0.8,
            tooltip=(f"<b>ZIP {row['zip']}</b><br>"
                     f"Hosp rate: {row['hosp_rate_adj']:.0f}<br>"
                     f"Readmit: {row['readmit_rate']:.1f}%<br>"
                     f"SVI: {row['svi_score']:.2f}<br>"
                     f"Discharges: {int(row['n_discharges'])}")
        ).add_to(m)

    m.save('/tmp/mod02/health_disparities_map.html')
    print("Interactive map saved: /tmp/mod02/health_disparities_map.html")
    print("Open in browser to explore — hover over circles for ZIP-level details.")
else:
    print("folium not available. Install with: pip install folium")
    print("The static scatter-map (Section 4) is the fallback visualisation.")


## 8. Knowledge Check
1. Why do we use age-adjusted rates rather than crude rates when comparing hospitalisation across ZIP codes?
2. The SVI-hospitalisation Pearson r = +0.58. What does this tell you? What doesn't it tell you?
3. Why might rural areas show lower hospitalisation rates despite worse health outcomes (hint: access to care)?
4. What additional data source would you add to strengthen this analysis?
5. Re-run the disparity analysis stratified by urban/rural. Does the SVI gradient hold within each stratum?


In [ ]:
# Exercise 5 — SVI quintile disparity by urban/rural
for urb in ['Urban','Suburban','Rural']:
    sub = zip_merged[zip_merged['urban_rural']==urb].copy()
    if len(sub) < 5: continue
    sub['q'] = pd.qcut(sub['svi_score'], min(5,len(sub)//2), labels=False, duplicates='drop')
    tbl = sub.groupby('q')[['hosp_rate_adj','readmit_rate']].mean().round(1)
    print(f"\n{urb} — outcomes by SVI quintile:")
    display(tbl)


---
**Next → NB-08: Capstone EDA — 30-Day Readmission**